# PolaFusion — Macro-F1-Aware Augmentation

Cells 1–3: Build the augmentation plan (which language-label pairs need synthetic data)  
Cells 4–6: Call Qwen3-235B via NVIDIA NIM to generate the synthetic examples  

**Run order:** augmentation_plan → augmentation_prompt for each subtask

In [ ]:
# augmentation plan subtask-1
import pandas as pd

# =========================
# PATH
# =========================
PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask1_full_train.csv"

df = pd.read_csv(PATH)

LANG_COL = "language"
LABEL_COL = "polarization"  # 1 = polarized, 0 = non-polarized

# =========================
# 🔴 PASTE YOUR F1 MACRO PER LANGUAGE HERE (from leaderboard output)
# =========================
f1_scores = {
    "amh": 0.7402,
    "arb": 0.8207,
    "ben": 0.8459,
    "deu": 0.7462,
    "eng": 0.7764,
    "fas": 0.8211,
    "hau": 0.7602,
    "hin": 0.7951,
    "ita": 0.6388,
    "khm": 0.6136,
    "mya": 0.8947,
    "nep": 0.85,
    "ori": 0.7425,
    "pan": 0.87,
    "pol": 0.8092,
    "rus": 0.7953,
    "spa": 0.7002,
    "swa": 0.8133,
    "tel": 0.839,
    "tur": 0.8347,
    "urd": 0.7549,
    "zho": 0.8971,
}

# =========================
# THRESHOLDS (tuned for Macro-F1)
# =========================
F1_THRESHOLD = 0.78
IMBALANCE_THRESHOLD = 0.30   # minority/majority ratio

rows = []

for lang in sorted(df[LANG_COL].unique()):
    lang_df = df[df[LANG_COL] == lang]

    pos = (lang_df[LABEL_COL] == 1).sum()
    neg = (lang_df[LABEL_COL] == 0).sum()

    if pos < neg:
        minority_label = 1
        minority_count = pos
        majority_count = neg
    else:
        minority_label = 0
        minority_count = neg
        majority_count = pos

    ratio = minority_count / majority_count if majority_count > 0 else 0
    f1 = f1_scores.get(lang, None)

    # Decide whether to augment
    do_augment = (
        f1 is not None and
        f1 < F1_THRESHOLD and
        ratio < IMBALANCE_THRESHOLD
    )

    target = min(int(0.8 * majority_count), minority_count * 2)
    to_generate = max(0, target - minority_count) if do_augment else 0

    rows.append({
        "language": lang,
        "F1_macro": f1,
        "polarized(1)": pos,
        "non_polarized(0)": neg,
        "minority_label_to_augment": minority_label,
        "minority/majority_ratio": round(ratio, 3),
        "augment?": do_augment,
        "samples_to_generate": to_generate
    })

plan_df = pd.DataFrame(rows)
plan_df.to_csv("subtask1_macroF1_aware_augmentation_plan.csv", index=False)

print("===== SUBTASK-1 MACRO-F1 AWARE AUGMENTATION PLAN =====")
print(plan_df)



In [ ]:
#augmentation plan subtask-2
import pandas as pd

DATA_PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask3_full_train.csv"

df = pd.read_csv(DATA_PATH)

LABEL_COLS = ["stereotype","vilification","dehumanization",
              "extreme_language","lack_of_empathy","invalidation"]

LANG_COL = "language"

# 🔴 PASTE YOUR PER-LABEL F1 HERE (from leaderboard)
f1_table = {
    "amh": {"dehumanization":0.5075,"lack_of_empathy":0.3087,"invalidation":0.3333},
    "arb": {"dehumanization":0.4762,"lack_of_empathy":0.5094,"invalidation":0.375},
    "ben": {"dehumanization":0.3146,"lack_of_empathy":0.0,"invalidation":0.0},
    "deu": {"dehumanization":0.4494,"lack_of_empathy":0.4643,"invalidation":0.3333},
    "eng": {"dehumanization":0.3871,"lack_of_empathy":0.3636,"invalidation":0.4932},
    "fas": {"dehumanization":0.3158,"lack_of_empathy":0.2326,"invalidation":0.2632},
    "hau": {"dehumanization":0.2,"lack_of_empathy":0.0,"invalidation":0.0},
    "khm": {"dehumanization":0.3333,"lack_of_empathy":0.4815,"invalidation":0.3288},
    "nep": {"dehumanization":0.6087,"lack_of_empathy":0.5,"invalidation":0.6364},
    "ori": {"dehumanization":0.0,"lack_of_empathy":0.0,"invalidation":0.5},
    "pan": {"dehumanization":0.5556,"lack_of_empathy":0.3226,"invalidation":0.5789},
    "spa": {"dehumanization":0.2078,"lack_of_empathy":0.4531,"invalidation":0.4483},
    "swa": {"dehumanization":0.3146,"lack_of_empathy":0.5691,"invalidation":0.4933},
    "tel": {"dehumanization":0.0,"lack_of_empathy":0.6292,"invalidation":0.4706},
    "tur": {"dehumanization":0.4583,"lack_of_empathy":0.3182,"invalidation":0.1333},
    "urd": {"dehumanization":0.7544,"lack_of_empathy":0.7511,"invalidation":0.7565},
    "zho": {"dehumanization":0.5806,"lack_of_empathy":0.4074,"invalidation":0.6154}
}

F1_THRESHOLD = 0.60
IMBALANCE_THRESHOLD = 0.25

rows = []

for lang in df[LANG_COL].unique():
    lang_df = df[df[LANG_COL] == lang]
    label_counts = lang_df[LABEL_COLS].sum()
    max_count = label_counts.max()

    for lab in ["dehumanization","lack_of_empathy","invalidation","extreme_language"]:
        count = label_counts.get(lab,0)
        ratio = count / max_count if max_count>0 else 0
        f1 = f1_table.get(lang,{}).get(lab,None)

        augment = (
            f1 is not None and
            f1 < F1_THRESHOLD and
            ratio < IMBALANCE_THRESHOLD and
            count > 0
        )

        target = min(int(0.8 * max_count), count * 2)
        to_generate = max(0, target - count) if augment else 0

        rows.append({
            "language": lang,
            "label": lab,
            "count": int(count),
            "max_label_count": int(max_count),
            "ratio": round(ratio,3),
            "F1": f1,
            "augment?": augment,
            "samples_to_generate": to_generate
        })

plan = pd.DataFrame(rows)
plan.to_csv("subtask3_macroF1_aware_augmentation_plan.csv", index=False)

print(plan[plan["augment?"]==True])


In [ ]:
#augmentation plan subtask-3
import pandas as pd

DATA_PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask2_full_train.csv"

df = pd.read_csv(DATA_PATH)

LABEL_COLS = ["political","racial/ethnic","religious","gender/sexual","other"]
LANG_COL = "language"

# 🔴 paste your per-label F1 here (from leaderboard table)
f1_table = {
    "amh": {"racial/ethnic":0.5636,"religious":0.4,"gender/sexual":0.0,"other":0.5743},
    "arb": {"racial/ethnic":0.5672,"religious":0.6,"gender/sexual":0.5532,"other":0.5231},
    "ben": {"racial/ethnic":0.0,"religious":0.25,"gender/sexual":0.5,"other":0.2778},
    "deu": {"racial/ethnic":0.6667,"religious":0.6087,"gender/sexual":0.5333,"other":0.1905},
    "eng": {"racial/ethnic":0.5263,"religious":0.4286,"gender/sexual":0.2857,"other":0.0},
    "fas": {"racial/ethnic":0.3636,"religious":0.619,"gender/sexual":0.4348,"other":0.6022},
    "hau": {"racial/ethnic":0.1667,"religious":0.5455,"gender/sexual":0.0,"other":0.0},
    "khm": {"racial/ethnic":0.2667,"religious":0.6667,"gender/sexual":0.625,"other":0.9036},
    "nep": {"racial/ethnic":0.8276,"religious":0.8889,"gender/sexual":0.7273,"other":0.4516},
    "ori": {"racial/ethnic":0.4,"religious":0.4,"gender/sexual":0.0,"other":0.0},
    "pan": {"racial/ethnic":0.125,"religious":0.381,"gender/sexual":0.4,"other":0.4667},
    "pol": {"racial/ethnic":0.7692,"religious":0.8571,"gender/sexual":0.3077,"other":0.375},
    "rus": {"racial/ethnic":0.5455,"religious":0.3077,"gender/sexual":0.6316,"other":0.0},
    "spa": {"racial/ethnic":0.5753,"religious":0.5357,"gender/sexual":0.7547,"other":0.4151},
    "swa": {"racial/ethnic":0.7744,"religious":0.7,"gender/sexual":0.2857,"other":0.3571},
    "tel": {"racial/ethnic":0.4872,"religious":0.2909,"gender/sexual":0.3784,"other":0.5517},
    "tur": {"racial/ethnic":0.68,"religious":0.6512,"gender/sexual":0.6154,"other":0.5},
    "urd": {"racial/ethnic":0.7378,"religious":0.7598,"gender/sexual":0.7182,"other":0.7215},
    "zho": {"racial/ethnic":0.8624,"religious":0.5,"gender/sexual":0.8158,"other":0.619}
}

F1_THRESHOLD = 0.60
IMBALANCE_THRESHOLD = 0.25

rows = []

for lang in df[LANG_COL].unique():
    lang_df = df[df[LANG_COL] == lang]
    label_counts = lang_df[LABEL_COLS].sum()
    max_count = label_counts.max()

    for lab in LABEL_COLS:
        count = label_counts[lab]
        ratio = count / max_count if max_count>0 else 0
        f1 = f1_table.get(lang,{}).get(lab,None)

        augment = (
            f1 is not None and
            f1 < F1_THRESHOLD and
            ratio < IMBALANCE_THRESHOLD and
            count > 0
        )

        target = min(int(0.8 * max_count), count * 2)
        to_generate = max(0, target - count) if augment else 0

        rows.append({
            "language": lang,
            "label": lab,
            "count": int(count),
            "max_label_count": int(max_count),
            "ratio": round(ratio,3),
            "F1": f1,
            "augment?": augment,
            "samples_to_generate": to_generate
        })

plan = pd.DataFrame(rows)
plan.to_csv("subtask2_macroF1_aware_augmentation_plan.csv", index=False)

print(plan[plan["augment?"]==True])


In [ ]:
#augmentation of subtask-1(prompt)
import pandas as pd
import time
from tqdm import tqdm
from openai import OpenAI

# =========================
# NVIDIA CLIENT
# =========================
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

MODEL_NAME = "qwen/qwen3-235b-a22b"

# =========================
# PATHS
# =========================
DATA_PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask1_full_train.csv"
PLAN_PATH = "/kaggle/input/sub1-plan/subtask1_macroF1_aware_augmentation_plan.csv"

LANG_COL = "language"
LABEL_COL = "polarization"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)
plan = pd.read_csv(PLAN_PATH)

# =========================
# PROMPT
# =========================
def build_prompt(text, label):
    lab_description = (
        "HIGHLY POLARIZED/BIASED. Keep the aggressive or divisive tone."
        if label == 1 else
        "NEUTRAL/NON-POLARIZED. Keep the objective or factual tone."
    )

    return f"""
[Task]
Act as a linguistic expert in social media discourse. Generate one synthetic variation of the text below for a machine learning dataset.

[Constraints]
- Label Context: The text is {lab_description}.
- Meaning: Preserve the same opinion, stance, and target.
- Linguistic Style: Maintain the original level of slang, intensity, and informal grammar.
- Content: Do NOT soften the language. Do NOT add apologies, disclaimers, or new entities.
- Output: Return ONLY the new sentence. No preamble.

[Text to Transform]
{text}

[Augmented Result]
"""

# =========================
# QWEN PARAPHRASE
# =========================
def qwen_paraphrase(prompt, retries=3):
    for _ in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                top_p=0.9,
                max_tokens=128,
                extra_body={"chat_template_kwargs": {"thinking": False}}
            )

            msg = resp.choices[0].message
            if msg.content:
                return msg.content.strip()

        except Exception as e:
            print("API error:", e)
            time.sleep(2)

    return None

# =========================
# AUGMENTATION LOOP
# =========================
aug_rows = []

for _, row in plan.iterrows():
    if not row["augment?"]:
        continue

    lang = row["language"]
    minority_label = int(row["minority_label_to_augment"])
    n_generate = int(row["samples_to_generate"])

    print(f"\nAugmenting language={lang}, label={minority_label}, samples={n_generate}")

    subset = df[(df[LANG_COL] == lang) & (df[LABEL_COL] == minority_label)]

    if len(subset) == 0:
        continue

    sampled = subset.sample(n=n_generate, replace=True, random_state=42)

    for _, r in tqdm(sampled.iterrows(), total=len(sampled), desc=f"{lang}-{minority_label}"):
        prompt = build_prompt(r["text"], minority_label)
        new_text = qwen_paraphrase(prompt)

        if new_text is None:
            continue

        new_row = r.copy()
        new_row["text"] = new_text
        aug_rows.append(new_row)

        time.sleep(0.25)  # rate control

# =========================
# SAVE
# =========================
df_aug = pd.concat([df, pd.DataFrame(aug_rows)], ignore_index=True)
df_aug = df_aug.sample(frac=1).reset_index(drop=True)

df_aug.to_csv("subtask1_augmented.csv", index=False)

print("\nSaved: subtask1_augmented.csv")
print("Original:", len(df))
print("Augmented:", len(df_aug))


In [ ]:
#augmentation of subtask-2(prompt)
import pandas as pd
import time
from tqdm import tqdm
from openai import OpenAI

# =========================
# NVIDIA CLIENT
# =========================
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

MODEL_NAME = "qwen/qwen3-235b-a22b"

# =========================
# PATHS
# =========================
DATA_PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask3_full_train.csv"
PLAN_PATH = "/kaggle/input/sub3-plan/subtask3_macroF1_aware_augmentation_plan.csv"

# =========================
# CONFIG
# =========================
MAX_PER_PAIR = 300
LANG_COL = "language"

LABEL_COLS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

# =========================
# LABEL DEFINITIONS (HINTS)
# =========================
manifestation_hints = {
    "stereotype": "Use generalized claims about a group, presenting them as inherent traits.",
    "vilification": "Depict the target as immoral, corrupt, or evil.",
    "dehumanization": "Portray the target as animals, objects, or less than human.",
    "extreme_language": "Use highly aggressive, exaggerated, or violent expressions.",
    "lack_of_empathy": "Show indifference or disregard for the suffering or feelings of the target.",
    "invalidation": "Dismiss or belittle the target’s experiences, opinions, or identity."
}

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)
plan = pd.read_csv(PLAN_PATH)

# =========================
# PROMPT (MULTI-LABEL SAFE)
# =========================
def build_prompt(text, labels, language_name):
    if isinstance(labels, list):
        label_str = ", ".join(labels)
        active_hints = " ".join([manifestation_hints.get(l, "") for l in labels])
    else:
        label_str = labels
        active_hints = manifestation_hints.get(labels, "Preserve the specific manifestation.")

    return f"""
[Task]
Act as a linguistic expert in social media discourse specialized in {language_name}.
Generate one synthetic variation of the {language_name} text below.

[Context]
This text expresses: {label_str}.
Requirement: {active_hints}

[Constraints]
- Preserve the same meaning, stance, and target.
- Maintain the same level of aggression and intensity.
- Do NOT sanitize, soften, or neutralize the language.
- Do NOT add or remove entities or targets.
- Do NOT switch to another manifestation type.
- Do NOT use words like "please", "respectfully", or "balanced".
- Do NOT replace slurs with generic insults.
- Keep slang and informal grammar if present.
- Output ONLY the new sentence. No explanations.

[Text to Transform]
{text}

[Augmented Result]
"""


# =========================
# QWEN CALL
# =========================
def qwen_paraphrase(prompt, retries=3):
    for _ in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                top_p=0.9,
                max_tokens=128,
                extra_body={"chat_template_kwargs": {"thinking": False}}
            )
            msg = resp.choices[0].message
            if msg.content:
                return msg.content.strip()
        except Exception as e:
            print("API error:", e)
            time.sleep(2)
    return None

# =========================
# =========================
# AUGMENTATION LOOP
# =========================
aug_rows = []

for _, row in plan.iterrows():
    if not row["augment?"]:
        continue

    lang = row["language"]
    target_label = row["label"]
    n_generate = min(int(row["samples_to_generate"]), MAX_PER_PAIR)

    print(f"\nAugmenting language={lang}, label={target_label}, samples={n_generate}")

    # Ensure we only sample from the correct language and where the label is present
    subset = df[(df[LANG_COL] == lang) & (df[target_label] == 1)]
    
    if len(subset) == 0 or n_generate == 0:
        print(f"Skipping {lang}-{target_label}: No seed data found.")
        continue

    sampled = subset.sample(n=n_generate, replace=True, random_state=42)

    for _, r in tqdm(sampled.iterrows(), total=len(sampled), desc=f"{lang}-{target_label}"):
        # 1. Identify all labels that are active for this specific row
        active_labels = [l for l in LABEL_COLS if r[l] == 1]
        
        # 2. FIX: Pass all THREE required arguments (text, labels, and language_name)
        prompt = build_prompt(r["text"], active_labels, lang)

        # 3. Call the model
        new_text = qwen_paraphrase(prompt)
        
        # 4. Filter out failures or safety refusals
        if new_text is None or len(new_text) < 5:
            continue

        # 5. Create the new row
        new_row = r.copy()
        new_row["text"] = new_text
        aug_rows.append(new_row)

        # Rate limiting sleep
        time.sleep(0.2)
# =========================
# SAVE DATASET
# =========================
df_aug = pd.concat([df, pd.DataFrame(aug_rows)], ignore_index=True)
df_aug = df_aug.sample(frac=1).reset_index(drop=True)

df_aug.to_csv("subtask3_augmented.csv", index=False)

print("\nSaved: subtask3_augmented.csv")
print("Original size:", len(df))
print("Augmented size:", len(df_aug))


In [ ]:
#augmentation of subtask-3(prompt)
import pandas as pd
import time
from tqdm import tqdm
from openai import OpenAI

# =========================
# NVIDIA CLIENT
# =========================
# Note: Ensure you rotate this key as discussed!
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

MODEL_NAME = "qwen/qwen3-235b-a22b"

# =========================
# PATHS
# =========================
DATA_PATH = "/kaggle/input/semeval-combined-dataset-evaluation-phase1/SEMEVAL/subtask2_full_train.csv"
PLAN_PATH = "/kaggle/input/sub2-plan/subtask2_macroF1_aware_augmentation_plan.csv"

# =========================
# CONFIG
# =========================
MAX_PER_PAIR = 300
LANG_COL = "language"
LABEL_COLS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]

# =========================
# LABEL DEFINITIONS (HINTS)
# =========================
type_hints = {
    "political": "Maintain political ideology, party bias, or political hostility.",
    "racial/ethnic": "Keep the bias/hostility toward the specific race, ethnicity, or nationality.",
    "religious": "Preserve the hostility or blame directed at the religion or religious group.",
    "gender/sexual": "Maintain the bias or insult regarding gender or sexual orientation.",
    "other": "Preserve targets like media, economy, or technology not covered by identity labels."
}

# =========================
# PROMPT (FIXED & IMPROVED)
# =========================
def build_prompt(text, labels, language_name):
    # Handle multi-label hints by creating a bulleted list for the AI
    if isinstance(labels, list):
        label_str = ", ".join(labels)
        active_hints = "\n".join([f"- {l.upper()}: {type_hints.get(l, '')}" for l in labels])
    else:
        label_str = labels
        active_hints = f"- {labels.upper()}: {type_hints.get(labels, 'Preserve category.')}"

    return f"""
[Task]
Act as a linguistic expert in social media discourse specialized in {language_name}. 
Generate one synthetic variation of the {language_name} text below.

[Context]
This text is polarized based on: {label_str}.
Requirements for labels:
{active_hints}

[Constraints]
- Preserve the same meaning, stance, and target.
- Maintain the same level of aggression and tone.
- Do NOT sanitize, soften, or neutralize the language.
- Do NOT add or remove entities or targets.
- Do NOT change the polarization category.
- Do NOT use words like "please", "respectfully", or "balanced".
- Output ONLY the new sentence. No explanation.

[Text to Transform]
{text}

[Augmented Result]
"""

# =========================
# QWEN CALL
# =========================
def qwen_paraphrase(prompt, retries=3):
    for _ in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                top_p=0.9,
                max_tokens=150, # Slightly higher for diverse languages
                extra_body={"chat_template_kwargs": {"thinking": False}}
            )
            msg = resp.choices[0].message
            if msg.content:
                content = msg.content.strip()
                # Basic safety check to catch AI refusals
                if "I cannot" in content or "policy" in content:
                    return None
                return content
        except Exception as e:
            print("API error:", e)
            time.sleep(2)
    return None

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)
plan = pd.read_csv(PLAN_PATH)

# =========================
# AUGMENTATION LOOP
# =========================
aug_rows = []

for _, row in plan.iterrows():
    if not row["augment?"]:
        continue

    lang = row["language"]
    target_label = row["label"]
    n_generate = min(int(row["samples_to_generate"]), MAX_PER_PAIR)

    print(f"\nAugmenting language={lang}, label={target_label}, samples={n_generate}")

    subset = df[(df[LANG_COL] == lang) & (df[target_label] == 1)]
    if len(subset) == 0 or n_generate == 0:
        continue

    # Sample with replacement to hit target numbers
    sampled = subset.sample(n=n_generate, replace=True, random_state=42)

    for _, r in tqdm(sampled.iterrows(), total=len(sampled), desc=f"{lang}-{target_label}"):
        active_labels = [l for l in LABEL_COLS if r[l] == 1]
        
        # FIX: Now passing 'lang' into the prompt function
        prompt = build_prompt(r["text"], active_labels, lang)

        new_text = qwen_paraphrase(prompt)
        
        # Ensure we don't save empty results or original text duplicates
        if new_text is None or new_text.lower() == r["text"].lower():
            continue

        new_row = r.copy()
        new_row["text"] = new_text
        aug_rows.append(new_row)

        time.sleep(0.2) # Adjusted rate limit

# =========================
# SAVE DATASET
# =========================
if aug_rows:
    df_aug = pd.concat([df, pd.DataFrame(aug_rows)], ignore_index=True)
    # Shuffle the final dataset to mix original and synthetic data
    df_aug = df_aug.sample(frac=1).reset_index(drop=True)
    df_aug.to_csv("subtask2_augmented.csv", index=False)
    print("\nSaved: subtask2_augmented.csv")
else:
    print("\nNo rows were augmented.")

print("Original size:", len(df))
print("Augmented size:", len(df_aug) if aug_rows else len(df))